# シュワルツシルト計量のリーマン曲率

静止した球対称質量$M$の外部で、次のようにおきます。

$$
A(r)=1-\frac{2GM}{c^2r}.
$$

符号$(+---)$、座標$(t,r,\theta,\phi)$を用いると、線素は

$$
ds^2=A\,dt^2-A^{-1}dr^2-r^2d\theta^2-r^2\sin^2\theta\,d\phi^2.
$$

この計量はリッチ平坦ですが、リーマン平坦ではありません。曲率の
全成分を計算する高負荷な展開を行わず、いくつかの成分を選んで
この違いを確認します。


## 局所フレーム、計量、逆計量

この座標表示は、$r>0$かつ$A=0$でない領域を覆います。曲面
$r=2GM/c^2$はこの座標表示における座標地平面ですが、$r=0$の
特異性は曲率不変量によって検出されます。
$\eta=\operatorname{diag}(1,-1,-1,-1)$である局所Lorentzフレームでは、
座標接ベクトルの成分は

$$
e_t=(\sqrt A,0,0,0),\quad
e_r=(0,A^{-1/2},0,0),\quad
e_\theta=(0,0,r,0),\quad
e_\phi=(0,0,0,r\sin\theta).
$$

Egisonは計量や逆計量を成分ごとに入力するのではなく、
$g_{ij}=\eta(e_i,e_j)$として計量を求め、その計量から逆計量を計算します。


In [1]:
declare symbol G, M, c, t, r, θ, φ: MathValue

def x : Vector MathValue := [| t, r, θ, φ |]
def A : MathValue := `(c^2 * r - 2 * G * M) / (c^2 * r)

def e_i_j : Matrix MathValue :=
  [| [| sqrt A, 0, 0, 0 |]
   , [| 0, 1 / sqrt A, 0, 0 |]
   , [| 0, 0, r, 0 |]
   , [| 0, 0, 0, r * sin θ |]
   |]_i_j

def minkowskiDot (u : Vector MathValue) (v : Vector MathValue) : MathValue :=
  u_1 * v_1 - u_2 * v_2 - u_3 * v_3 - u_4 * v_4

def g_i_j : Matrix MathValue :=
  generateTensor (\[i, j] -> minkowskiDot e_i_# e_j_#) [4, 4]

def g~i~j : Matrix MathValue := M.inverse g_#_#


In [2]:
A


$(c^{2} r - 2 G M) c^{-2} r^{-1}$

In [3]:
g_#_#


$\begin{pmatrix} (c^{2} r - 2 G M) c^{-2} r^{-1} & 0 & 0 & 0 \\ 0 & -(c^{2} r - 2 G M)^{-1} c^{2} r & 0 & 0 \\ 0 & 0 & -r^{2} & 0 \\ 0 & 0 & 0 & -\sin(θ)^{2} r^{2} \\ \end{pmatrix}_{\#\#}^{\;\;}$

In [4]:
g~#~#


$\begin{pmatrix} (c^{2} r - 2 G M)^{-1} c^{2} r & 0 & 0 & 0 \\ 0 & -(c^{2} r - 2 G M) c^{-2} r^{-1} & 0 & 0 \\ 0 & 0 & -r^{-2} & 0 \\ 0 & 0 & 0 & -\sin(θ)^{-2} r^{-2} \\ \end{pmatrix}_{\;\;}^{\#\#}$

## レヴィ・チヴィタ接続

接続を表として入力するのではなく、計量から計算します。
$\Gamma^\theta{}_{r\theta}=1/r$や
$\Gamma^\phi{}_{\theta\phi}=\cot\theta$などの角度方向の成分は、
特に簡潔な検算になります。


In [5]:
def Γ_i_j_k : Tensor MathValue :=
  (1 / 2) * (∂/∂ g_i_k x~j + ∂/∂ g_i_j x~k - ∂/∂ g_j_k x~i)

def Γ~i_j_k : Tensor MathValue := withSymbols [m]
  g~i~m . Γ_m_j_k


In [6]:
Γ~3_2_3


$r^{-1}$

In [7]:
Γ~4_3_4


$\cos(θ) \sin(θ)^{-1}$

## リーマンテンソルとリッチテンソル

球面のNotebookと同じ符号規約を用います。$M\ne0$のとき、角度成分
$R^\theta{}_{\phi\theta\phi}$は0ではありません。平坦な球座標計量では、
その動径方向と角度方向の項が打ち消し合います。リッチ曲率は
$R^m{}_{imj}$を縮約したものです。


In [8]:
def R~i_j_k_l : Tensor MathValue := withSymbols [m]
  expandAll
    (∂/∂ Γ~i_j_l x~k - ∂/∂ Γ~i_j_k x~l
      + Γ~m_j_l . Γ~i_m_k - Γ~m_j_k . Γ~i_m_l)

def Ric_i_j : Matrix MathValue := withSymbols [m]
  sum (contract R~m_i_m_j)

def scalarCurvature : MathValue := withSymbols [i, j]
  g~i~j . Ric_i_j


In [9]:
expandAll (R~3_4_3_4)


$2 \sin(θ)^{2} G M c^{-2} r^{-1}$

In [10]:
expandAll (Ric_1_1)


$0$

## 物理的・幾何学的な意味

リーマンテンソルの成分は潮汐曲率を捉えます。一方、リッチテンソルの
成分は、真空アインシュタイン方程式が要請するとおり0に簡約されます。
座標に依存しないクレッチマン不変量は

$$
R_{abcd}R^{abcd}=\frac{48G^2M^2}{c^4r^6}.
$$

です。したがって、曲率はシュワルツシルト地平面では有限ですが、
$r=0$では発散します。4つの添字の全組合せを展開すると対話的なデモには
不必要なほど計算量が大きくなるため、不変量の完全な縮約は出力セルで
計算せず、式として示しています。
